In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import requests  # HTTP library for Python
import bs4
import math
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import gsw
%matplotlib inline
import os
import sys
import xmitgcm 
import xgcm
import pyproj
#import wget
#sys.path.append('../../ECCOv4-py/ECCOv4-py')
#import ecco_v4_py as ecco
import cmocean
sys.path.append('/home/basil/Desktop/cape_mend_SM_LCS/datatools')
from datatools import datatools as tools
#from moviepy.config import change_settings
#change_settings({"FFMPEG_BINARY": "/usr/bin/ffmpeg"})
#import moviepy.video.io.ImageSequenceClip 

from scipy.fft import fft, fftfreq
from scipy.signal import periodogram
from scipy.signal import welch
from scipy.stats import chi2
from opendrift.readers import reader_netCDF_CF_generic
from opendrift.readers import reader_global_landmask
import opendrift.readers as readers
from opendrift.models.oceandrift import OceanDrift
from datetime import timedelta
from scipy.interpolate import griddata
from tqdm.notebook import tqdm

In [2]:
path_tohd = '/media/basil/Elements/data/Thesis/LLC4320/nc_files/'
grid_path = '/home/basil/Desktop/cape_mend_SM_LCS/data/cape_mend_grid_llc4320.nc'
grid = xr.open_dataset(grid_path)
variables = ['Eta','Theta','Salt','U','V','W','oceTAUX','oceTAUY']

In [5]:
taux_files, taux_filepaths = tools.get_data_paths_from_binary(path_tohd,variables[-2],file_end='nc')
tauy_files, tauy_filepaths = tools.get_data_paths_from_binary(path_tohd,variables[-1],file_end='nc')

tauxy = xr.open_mfdataset(taux_filepaths+tauy_filepaths)


In [4]:
taux_filepaths

['/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000179712.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000179856.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180000.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180144.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180288.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180432.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180576.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180720.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000180864.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000181008.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceTAUX_0000181152.nc',
 '/media/basil/Elements/data/Thesis/LLC4320/nc_files/oceTAUX/oceT

In [6]:
tauxy = xr.merge([tauxy,grid]).set_coords({'XC','YC'})

In [7]:
extent = [-127.59791666666666,-123.10208333333333,38.50208333333333,41.99791666666667]
ds = tauxy
ds_cut = ds.where(((ds['XC']>extent[0])&(ds['XC']<extent[1]))&(ds['YC']>extent[2])&(ds['YC']<extent[3]))
ds_cut_tau = ds_cut.dropna(how='all',dim='i').dropna(how='all',dim='j') #this code takes forever

In [ ]:
cutout_path = '/media/basil/Elements/data/Thesis/LLC4320/nc_files/cutouts/'

ds_cut_tau.to_netcdf(cutout_path+'tau_cape_mend.nc')

In [10]:
tauxy

<xarray.Dataset>
Dimensions:  (time: 9119, i: 1486, j: 672, k: 1)
Coordinates:
  * time     (time) datetime64[ns] 2011-11-01 ... 2012-11-14T23:00:00
  * i        (i) int64 0 1 2 3 4 5 6 7 ... 1479 1480 1481 1482 1483 1484 1485
  * j        (j) int64 0 1 2 3 4 5 6 7 8 ... 663 664 665 666 667 668 669 670 671
    XC       (i, j) float32 -132.0 -132.0 -131.9 -131.9 ... -118.1 -118.0 -118.0
    YC       (i, j) float32 28.0 28.0 28.0 28.0 28.0 ... 49.99 49.99 49.99 49.99
  * k        (k) int64 0
Data variables: (12/13)
    oceTAUY  (time, i, j) float32 dask.array<chunksize=(1, 1486, 672), meta=np.ndarray>
    DXC      (i, j) float32 ...
    DXG      (i, j) float32 ...
    DXV      (i, j) float32 ...
    DYC      (i, j) float32 ...
    DYG      (i, j) float32 ...
    ...       ...
    Depth    (i, j) float32 ...
    RAC      (i, j) float32 ...
    RAZ      (i, j) float32 ...
    XG       (i, j) float32 ...
    YG       (i, j) float32 ...
    hFacC    (i, j, k) float32 ...